# Auditoria Operativa del Modelo

Este notebook permite auditar el flujo operativo del proyecto y registrar nuevos llamados reales al modelo.

Objetivos principales:
- Validar que un payload JSON cumple la estructura esperada.
- Ejecutar inferencia local con los artefactos reales del proyecto.
- Registrar nuevos ejemplos reales en `json_examples/m07` o `json_examples/m09`.
- Actualizar `json_examples/labels_manifest.csv` sin introducir datos sinteticos.
- Generar artefactos de trazabilidad para auditoria.

Regla de auditoria:
- No registrar datos sinteticos, artificiales, interpolados o aumentados.

## 1. Configuracion de trabajo

Ejecute esta seccion primero. Define rutas, reglas de negocio y convenciones de registro.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import re
import shutil
from datetime import datetime
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
JSON_EXAMPLES_DIR = PROJECT_DIR / "json_examples"
LABELS_MANIFEST_PATH = JSON_EXAMPLES_DIR / "labels_manifest.csv"
AUDIT_OUTPUT_DIR = PROJECT_DIR / "resultados_capitulo_6" / "auditoria_operativa"
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXERCISES = {
    "m07": {"exercise_id": 7, "descripcion": "Rotacion interna/externa de hombro"},
    "m09": {"exercise_id": 9, "descripcion": "Abduccion de hombro"},
}

MODELS_CONFIG = {
    7: {
        "module": PROJECT_DIR / "standing_shoulder_internal_external_rotation" / "Standing_Shoulder_Internal_External_Rotation.py",
        "bundle": PROJECT_DIR / "standing_shoulder_internal_external_rotation" / "standing_shoulder_internal_external_rotation_artifacts" / "standing_shoulder_internal_external_rotation_bundle.joblib",
        "model": PROJECT_DIR / "standing_shoulder_internal_external_rotation" / "standing_shoulder_internal_external_rotation_artifacts" / "standing_shoulder_internal_external_rotation_tcn.keras",
    },
    9: {
        "module": PROJECT_DIR / "standing_shoulder_abduction" / "Standing_Shoulder_Abduction.py",
        "bundle": PROJECT_DIR / "standing_shoulder_abduction" / "standing_shoulder_abduction_artifacts" / "standing_shoulder_abduction_bundle.joblib",
        "model": PROJECT_DIR / "standing_shoulder_abduction" / "standing_shoulder_abduction_artifacts" / "standing_shoulder_abduction_tcn.keras",
    },
}

FORBIDDEN_NAME_PATTERN = re.compile(r"sintet|synthetic|augment|augmented|artificial", re.IGNORECASE)
SEQUENCE_KEYS = ("secuencia", "frames", "ventana", "window")

print("PROJECT_DIR:", PROJECT_DIR)
print("JSON_EXAMPLES_DIR:", JSON_EXAMPLES_DIR)
print("LABELS_MANIFEST_PATH:", LABELS_MANIFEST_PATH)
print("AUDIT_OUTPUT_DIR:", AUDIT_OUTPUT_DIR)

## 2. Utilidades de auditoria

Estas funciones permiten validar payloads, cargar el runtime real, ejecutar inferencia y registrar nuevos ejemplos.

In [ ]:
RUNTIME_CACHE: dict[int, tuple[Any, Any, dict[str, Any]]] = {}

def _load_module(module_path: Path, module_name: str) -> Any:
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"No se pudo cargar modulo: {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def parse_exercise_id(payload: dict[str, Any]) -> int:
    raw = payload.get("ejercicio_id", payload.get("movement_id"))
    if isinstance(raw, int):
        return raw
    if isinstance(raw, str):
        value = raw.strip().lower().replace("m", "")
        if value.isdigit():
            return int(value)
    raise ValueError("No se pudo determinar el ejercicio. Use ejercicio_id o movement_id con m07/m09 o 7/9.")

def exercise_code_from_id(exercise_id: int) -> str:
    for code, meta in VALID_EXERCISES.items():
        if meta["exercise_id"] == exercise_id:
            return code
    raise ValueError(f"Ejercicio no soportado: {exercise_id}")

def load_runtime(exercise_id: int) -> tuple[Any, Any, dict[str, Any]]:
    if exercise_id in RUNTIME_CACHE:
        return RUNTIME_CACHE[exercise_id]

    conf = MODELS_CONFIG.get(exercise_id)
    if conf is None:
        raise ValueError("Ejercicio no soportado. Use m07 o m09.")

    module = _load_module(conf["module"], f"audit_runtime_{exercise_id}")
    bundle = joblib.load(conf["bundle"])
    tf = module.require_tensorflow()
    model = tf.keras.models.load_model(conf["model"])

    runtime = (module, model, bundle)
    RUNTIME_CACHE[exercise_id] = runtime
    return runtime

def read_json_payload(json_path: Path) -> dict[str, Any]:
    with json_path.open("r", encoding="utf-8") as fh:
        return json.load(fh)

def validate_payload(payload: dict[str, Any], file_name: str | None = None) -> dict[str, Any]:
    if not isinstance(payload, dict):
        raise ValueError("El payload debe ser un objeto JSON.")

    if file_name and FORBIDDEN_NAME_PATTERN.search(file_name):
        raise ValueError("El nombre del archivo sugiere datos sinteticos o artificiales. No es apto para auditoria.")

    sequence_key = next((key for key in SEQUENCE_KEYS if key in payload), None)
    if sequence_key is None:
        raise ValueError("El payload no contiene secuencia. Se esperaba una de estas claves: secuencia, frames, ventana, window.")

    sequence = payload[sequence_key]
    if not isinstance(sequence, list) or len(sequence) == 0:
        raise ValueError("La secuencia debe ser una lista no vacia.")

    exercise_id = parse_exercise_id(payload)
    exercise_code = exercise_code_from_id(exercise_id)

    return {
        "exercise_id": exercise_id,
        "exercise_code": exercise_code,
        "sequence_key": sequence_key,
        "num_frames": len(sequence),
    }

def infer_local(payload: dict[str, Any]) -> dict[str, Any]:
    validation = validate_payload(payload)
    module, model, bundle = load_runtime(validation["exercise_id"])
    frames = module.coerce_frames(payload)
    prediction = module.infer_from_window(frames_json=frames, model=model, bundle=bundle)
    if not isinstance(prediction, dict):
        raise RuntimeError("La inferencia no devolvio un diccionario utilizable.")
    return {**validation, **prediction}

def load_labels_manifest() -> pd.DataFrame:
    if LABELS_MANIFEST_PATH.exists():
        df = pd.read_csv(LABELS_MANIFEST_PATH)
    else:
        df = pd.DataFrame(columns=["exercise", "file", "label", "suggested_label", "has_movement_id", "notes"])
    for col in ["exercise", "file", "notes"]:
        if col in df.columns:
            df[col] = df[col].astype(str).replace("nan", "")
    return df

def next_example_name(exercise_code: str) -> str:
    target_dir = JSON_EXAMPLES_DIR / exercise_code
    target_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(target_dir.glob("example_*.json"))
    if not existing:
        return "example_001.json"
    numbers = []
    for path in existing:
        match = re.match(r"example_(\d+)\.json$", path.name)
        if match:
            numbers.append(int(match.group(1)))
    next_num = (max(numbers) + 1) if numbers else 1
    return f"example_{next_num:03d}.json"

def register_new_example(
    source_json_path: str | Path,
    expected_label: int | None = None,
    notes: str = "",
    overwrite: bool = False,
) -> dict[str, Any]:
    source_path = Path(source_json_path)
    if not source_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {source_path}")

    if FORBIDDEN_NAME_PATTERN.search(source_path.name):
        raise ValueError("No se puede registrar un archivo con nombre asociado a datos sinteticos o artificiales.")

    payload = read_json_payload(source_path)
    validation = validate_payload(payload, file_name=source_path.name)
    exercise_code = validation["exercise_code"]

    target_name = next_example_name(exercise_code)
    target_path = JSON_EXAMPLES_DIR / exercise_code / target_name
    target_path.parent.mkdir(parents=True, exist_ok=True)

    if target_path.exists() and not overwrite:
        raise FileExistsError(f"El archivo destino ya existe: {target_path}")

    shutil.copy2(source_path, target_path)

    manifest_df = load_labels_manifest()
    new_row = {
        "exercise": exercise_code,
        "file": target_name,
        "label": expected_label if expected_label in (0, 1) else np.nan,
        "suggested_label": expected_label if expected_label in (0, 1) else np.nan,
        "has_movement_id": 1 if payload.get("movement_id") is not None or payload.get("ejercicio_id") is not None else 0,
        "notes": notes or f"Alta desde notebook de auditoria {datetime.now().isoformat(timespec='seconds')}",
    }
    manifest_df = pd.concat([manifest_df, pd.DataFrame([new_row])], ignore_index=True)
    manifest_df.to_csv(LABELS_MANIFEST_PATH, index=False)

    trace = {
        "source_path": str(source_path),
        "registered_path": str(target_path),
        "exercise_code": exercise_code,
        "label": expected_label,
        "notes": new_row["notes"],
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    trace_path = AUDIT_OUTPUT_DIR / f"alta_{exercise_code}_{target_name.replace('.json', '')}.json"
    trace_path.write_text(json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8")
    return trace

def audit_inventory() -> pd.DataFrame:
    manifest_df = load_labels_manifest()
    rows = []
    for exercise_code in sorted(VALID_EXERCISES):
        files = sorted((JSON_EXAMPLES_DIR / exercise_code).glob("*.json"))
        subset = manifest_df[manifest_df["exercise"].eq(exercise_code)] if "exercise" in manifest_df.columns else pd.DataFrame()
        rows.append({
            "exercise": exercise_code,
            "descripcion": VALID_EXERCISES[exercise_code]["descripcion"],
            "json_registrados": len(files),
            "filas_manifest": int(len(subset)),
            "labels_pendientes": int(subset["label"].isna().sum()) if len(subset) and "label" in subset.columns else 0,
        })
    inventory_df = pd.DataFrame(rows)
    inventory_df.to_csv(AUDIT_OUTPUT_DIR / "inventario_auditoria_operativa.csv", index=False)
    return inventory_df

## 3. Inventario inicial de auditoria

Esta celda muestra el estado actual del inventario de ejemplos y del manifiesto.

In [ ]:
inventory_df = audit_inventory()
manifest_df = load_labels_manifest()
display(inventory_df)
display(manifest_df.head(10))
print("Labels pendientes totales:", int(manifest_df["label"].isna().sum()) if "label" in manifest_df.columns else 0)

## 4. Seleccionar un JSON para revisar o inferir

Use `SOURCE_JSON_PATH` para apuntar a un archivo existente. Puede ser un archivo ya registrado en `json_examples` o un nuevo archivo externo que quiera auditar antes de darlo de alta.

In [ ]:
SOURCE_JSON_PATH = PROJECT_DIR / "jsonEjemplo.json"
SOURCE_JSON_PATH

## 5. Validacion estructural del payload

Esta celda no registra nada. Solo comprueba que el payload sea apto para revisión e inferencia.

In [ ]:
payload = read_json_payload(Path(SOURCE_JSON_PATH))
payload_validation = validate_payload(payload, file_name=Path(SOURCE_JSON_PATH).name)
payload_validation_df = pd.DataFrame([payload_validation])
payload_validation_df.to_csv(AUDIT_OUTPUT_DIR / "validacion_payload_actual.csv", index=False)
display(payload_validation_df)
print("Archivo apto para revision estructural.")

## 6. Inferencia local del modelo

Ejecuta el runtime real del proyecto sobre el payload seleccionado y guarda el resultado como evidencia operativa.

In [ ]:
inference_result = infer_local(payload)
inference_df = pd.DataFrame([inference_result])
timestamp_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
inference_csv_path = AUDIT_OUTPUT_DIR / f"inferencia_{timestamp_tag}.csv"
inference_json_path = AUDIT_OUTPUT_DIR / f"inferencia_{timestamp_tag}.json"
inference_df.to_csv(inference_csv_path, index=False)
inference_json_path.write_text(json.dumps(inference_result, ensure_ascii=False, indent=2), encoding="utf-8")
display(inference_df)
print("CSV de inferencia:", inference_csv_path)
print("JSON de inferencia:", inference_json_path)

## 7. Alta de un nuevo llamado real

Complete los parámetros y ejecute la celda solo si desea registrar el archivo como nuevo ejemplo auditado.

Reglas:
- Solo usar llamados reales.
- Si ya conoce la etiqueta clínica, usar `EXPECTED_LABEL = 0` o `1`.
- Si aún no la conoce, deje `EXPECTED_LABEL = None`; el registro quedará pendiente en `labels_manifest.csv`.

In [ ]:
REGISTER_NEW_EXAMPLE = False
EXPECTED_LABEL = None
REGISTRATION_NOTES = "Alta manual desde notebook de auditoria operativa"

if REGISTER_NEW_EXAMPLE:
    registration_trace = register_new_example(
        source_json_path=SOURCE_JSON_PATH,
        expected_label=EXPECTED_LABEL,
        notes=REGISTRATION_NOTES,
    )
    display(pd.DataFrame([registration_trace]))
else:
    print("Registro desactivado. Cambie REGISTER_NEW_EXAMPLE=True solo cuando quiera dar de alta un nuevo ejemplo real.")

## 8. Verificacion posterior al alta

Ejecute esta celda despues de un alta para confirmar inventario, manifiesto y pendientes.

In [ ]:
inventory_after_df = audit_inventory()
manifest_after_df = load_labels_manifest()
pending_after_df = manifest_after_df[manifest_after_df["label"].isna()].copy() if "label" in manifest_after_df.columns else pd.DataFrame()
pending_after_path = AUDIT_OUTPUT_DIR / "labels_pendientes_post_alta.csv"
pending_after_df.to_csv(pending_after_path, index=False)
display(inventory_after_df)
display(pending_after_df.head(20))
print("Pendientes exportados en:", pending_after_path)

## 9. Checklist de auditoria operativa

Use este checklist antes de considerar válido un nuevo llamado registrado.

1. El archivo fuente corresponde a una captura real.
2. El nombre del archivo no contiene referencias a datos sinteticos o artificiales.
3. El payload contiene `ejercicio_id` o `movement_id` con valor compatible con m07 o m09.
4. La secuencia está presente en `secuencia`, `frames`, `ventana` o `window`.
5. La inferencia local se ejecuta sin error sobre artefactos reales del proyecto.
6. El archivo se copia a la carpeta correcta dentro de `json_examples`.
7. `labels_manifest.csv` se actualiza con trazabilidad.
8. Si no existe etiqueta clínica definitiva, el registro queda pendiente y visible en el manifiesto.
9. Los archivos de evidencia quedan exportados en `resultados_capitulo_6/auditoria_operativa`.